In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cftime
import datetime
from datetime import date
from matplotlib import pyplot
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy
import pandas
from scipy.stats import ks_2samp, mannwhitneyu
import xarray as xr

In [2]:
# Define directories
Diri = '../ExtraTrack_Data/Output_Files_V7/'
Output_Diri = '../RCP_Figs/Analysis_Figs_V7.3.2/'

In [3]:
# Open file
def Open_File(File):
    DF = pandas.read_csv(File)
    DF = DF.drop("Unnamed: 0", axis=1)
    return (DF)

In [4]:
# Open each file
def Files_Open(Scenario, Diri, Subset):
    Data_DF = Open_File(Diri+Scenario+'_Data_'+Subset+'_Output.csv')
    ET_DF = Open_File(Diri+Scenario+'_ET_'+Subset+'_Output.csv')
    Codes_DF = Open_File(Diri+Scenario+'_Codes_Output.csv')
# Edit time format
    Time_Cols = ["ET Begin Time", "ET Complete Time", "Trop Peak Time", "Peak Time", "Genesis Time", "Final Time"]
    for Col in Time_Cols:
        ET_DF[Col] = pandas.to_datetime(ET_DF[Col], errors="coerce")
    Data_DF["Time(Z)"] = pandas.to_datetime(Data_DF["Time(Z)"], errors="coerce")
    return (Data_DF, ET_DF, Codes_DF)

In [5]:
# Create bins
def Create_Bins(Min, Max, Bin_Width):
    Bins = numpy.arange(Min, Max+Bin_Width, Bin_Width)
    return (Bins)
Lon_Bins = Create_Bins(-100,20,5)
Lat_Bins = Create_Bins(0,60,5)

In [6]:
# Number of years for each climate scenario
Num_Years = numpy.array([90,93,93])

In [7]:
# Open files
Control_Data, Control_ET, Control_Codes = Files_Open("Control", Diri, "SubsetB")
RCP45_Data, RCP45_ET, RCP45_Codes = Files_Open("RCP45", Diri, "SubsetB")
RCP85_Data, RCP85_ET, RCP85_Codes = Files_Open("RCP85", Diri, "SubsetB")

In [8]:
# Bootstrap test for statistical significance between medians
def Bootstrap_Test(A, B, n=10000, Random_Seed=28):
    Actual_Diff = numpy.median(B) - numpy.median(A)
    Boot_Diffs = numpy.zeros(n)
    rng = numpy.random.default_rng(Random_Seed)
# Bootstrap test
    for i in range(n):
        A_Sample = rng.choice(A, size=len(A), replace=True)
        B_Sample = rng.choice(B, size=len(B), replace=True)
        Boot_Diffs[i] = numpy.median(B_Sample) - numpy.median(A_Sample)
# Get confidence interval
    CI_Low, CI_High = numpy.percentile(Boot_Diffs[i], [2.5, 97.5])
# Approximate two sided P value
    if Actual_Diff >= 0:
        P_Val = 2 * numpy.mean(Boot_Diffs <= 0)
    else:
        P_Val = 2 * numpy.mean(Boot_Diffs >= 0)
    P_Val = numpy.clip(P_Val, 0, 1)
    return (P_Val)

In [9]:
# Create subsets for each storm phase
def Phase_Subsets(Data):
    Subset_0 = Data[Data["SLP(hPa)"] <= 1010].reset_index()
    Subset_Trop = Subset_0[Subset_0["Storm Phase"] == "Tropical"]
    Subset_Trans = Subset_0[(Subset_0["Storm Phase"] == "Transition")]
    Subset_Extra = Subset_0[Subset_0["Storm Phase"] == "Extratropical"]
    return (Subset_0, Subset_Trop, Subset_Trans, Subset_Extra)

In [10]:
# Find statistical significance for storm phase cumulative distributions
def Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, Var):
# Create empty arrays
    Medians = numpy.zeros((3,4))
    KS_P_Vals = numpy.zeros((2,4))
    MW_P_Vals = numpy.zeros((2,4))
    Boot_P_Vals = numpy.zeros((2,4))
    Phases = ["All", "Tropical", "Transitioning", "Extratropical"]
#
# Find statistical significance for each phase
    for i in range(4):
# Create subsets
        Control_Phase = Phase_Subsets(Control_Data)[i]
        RCP45_Phase = Phase_Subsets(RCP45_Data)[i]
        RCP85_Phase = Phase_Subsets(RCP85_Data)[i]
# Compute medians
        Medians[0][i] = numpy.median(Control_Phase[Var])
        Medians[1][i] = numpy.median(RCP45_Phase[Var])
        Medians[2][i] = numpy.median(RCP85_Phase[Var])
# Compute statistical significance with KS Test
        KS_P_Vals[0][i] = ks_2samp(Control_Phase[Var], RCP45_Phase[Var])[1]
        KS_P_Vals[1][i] = ks_2samp(Control_Phase[Var], RCP85_Phase[Var])[1]
# Compute statistical significance with Mann Whitney Test
        MW_P_Vals[0][i] = mannwhitneyu(Control_Phase[Var], RCP45_Phase[Var])[1]
        MW_P_Vals[1][i] = mannwhitneyu(Control_Phase[Var], RCP85_Phase[Var])[1]
# Compute statistical significance with bootstrap test
        Boot_P_Vals[0][i] = Bootstrap_Test(Control_Phase[Var], RCP45_Phase[Var])
        Boot_P_Vals[1][i] = Bootstrap_Test(Control_Phase[Var], RCP85_Phase[Var])
#
# Output as dataframe
    Signif_DF = pandas.DataFrame({"Variable": [Var, Var, Var, Var], "Phase": Phases, \
                "Median (Control)": Medians[0], "Median (RCP4.5)": Medians[1], "Median (RCP8.5)": Medians[2], \
                "KS Test (RCP4.5 - Control)": KS_P_Vals[0], "KS Test (RCP8.5 - Control)": KS_P_Vals[1], \
                "MW Test (RCP4.5 - Control)": MW_P_Vals[0], "MW Test (RCP8.5 - Control)": MW_P_Vals[1], \
                "Bootstrap Test (RCP4.5 - Control)": Boot_P_Vals[0], "Bootstrap Test (RCP8.5 - Control)": Boot_P_Vals[1]})
    Signif_DF = Signif_DF.round(3)
    return (Signif_DF)

In [11]:
# Function for applying SLP bounds
def ET_SLP_Bounds(Control_ET, RCP45_ET, RCP85_ET, Var, Low_Bound, Up_Bound):
    Control_ET_Bound = Control_ET[(Control_ET[Var] <= Up_Bound) & (Control_ET[Var] >= Low_Bound)].reset_index(drop=True)
    RCP45_ET_Bound = RCP45_ET[(RCP45_ET[Var] <= Up_Bound) & (RCP45_ET[Var] >= Low_Bound)].reset_index(drop=True)
    RCP85_ET_Bound = RCP85_ET[(RCP85_ET[Var] <= Up_Bound) & (RCP85_ET[Var] >= Low_Bound)].reset_index(drop=True)
    return (Control_ET_Bound, RCP45_ET_Bound, RCP85_ET_Bound)

In [18]:
# Find statistical significance for ET cumulative distributions
def ET_Diff_Signif(Control_ET, RCP45_ET, RCP85_ET, Var):
# Create empty arrays
    Medians = numpy.zeros((3,5))
    KS_P_Vals = numpy.zeros((2,5))
    MW_P_Vals = numpy.zeros((2,5))
    Boot_P_Vals = numpy.zeros((2,5))
#
# Find statistical significance for genesis, peak, tropical peak, ET initiation and ET completion
    Snaps = ["Genesis", "Peak", "Trop Peak", "ET Begin", "ET Complete"]
    SLP_Bounds = [1013, 1000, 1000, 1010, 1010]
    for i in range(5):
# Create subsets
        Vari = Snaps[i] + " " + Var
        Var_SLP = Snaps[i] + " SLP"
        Control_ET_Subset, RCP45_ET_Subset, RCP85_ET_Subset = \
        ET_SLP_Bounds(Control_ET, RCP45_ET, RCP85_ET, Var_SLP, 728, SLP_Bounds[i])
# Compute medians
        Medians[0][i] = numpy.median(Control_ET_Subset[Vari])
        Medians[1][i] = numpy.median(RCP45_ET_Subset[Vari])
        Medians[2][i] = numpy.median(RCP85_ET_Subset[Vari])
# Compute statistical significance with KS Test
        KS_P_Vals[0][i] = ks_2samp(Control_ET_Subset[Vari], RCP45_ET_Subset[Vari])[1]
        KS_P_Vals[1][i] = ks_2samp(Control_ET_Subset[Vari], RCP85_ET_Subset[Vari])[1]
# Compute statistical significance with Mann Whitney Test
        MW_P_Vals[0][i] = mannwhitneyu(Control_ET_Subset[Vari], RCP45_ET_Subset[Vari])[1]
        MW_P_Vals[1][i] = mannwhitneyu(Control_ET_Subset[Vari], RCP85_ET_Subset[Vari])[1]
# Compute statistical significance with bootstrap test
        Boot_P_Vals[0][i] = Bootstrap_Test(Control_ET_Subset[Vari], RCP45_ET_Subset[Vari])
        Boot_P_Vals[1][i] = Bootstrap_Test(Control_ET_Subset[Vari], RCP85_ET_Subset[Vari])
#
# Output as dataframe
    Signif_DF = pandas.DataFrame({"Variable": [Var, Var, Var, Var, Var], "Snapshot": Snaps, \
                "Median (Control)": Medians[0], "Median (RCP4.5)": Medians[1], "Median (RCP8.5)": Medians[2], \
                "KS Test (RCP4.5 - Control)": KS_P_Vals[0], "KS Test (RCP8.5 - Control)": KS_P_Vals[1], \
                "MW Test (RCP4.5 - Control)": MW_P_Vals[0], "MW Test (RCP8.5 - Control)": MW_P_Vals[1], \
                "Bootstrap Test (RCP4.5 - Control)": Boot_P_Vals[0], "Bootstrap Test (RCP8.5 - Control)": Boot_P_Vals[1]})
    Signif_DF = Signif_DF.round(3)
    return (Signif_DF)

In [13]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "SLP(hPa)")

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,SLP(hPa),All,990.380,989.435,988.05,0.000,0.000,0.002,0.0,0.014,0.000
1,SLP(hPa),Tropical,991.480,990.670,989.20,0.000,0.000,0.020,0.0,0.069,0.000
2,SLP(hPa),Transitioning,975.095,969.800,971.51,0.000,0.000,0.000,0.0,0.001,0.002
3,SLP(hPa),Extratropical,993.610,992.740,991.39,0.004,0.003,0.053,0.0,0.489,0.005


In [14]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "Lat")

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,Lat,All,30.53,32.24,32.66,0.000,0.00,0.000,0.000,0.000,0.000
1,Lat,Tropical,26.73,28.59,29.17,0.000,0.00,0.000,0.000,0.000,0.000
2,Lat,Transitioning,40.24,41.23,40.76,0.012,0.02,0.001,0.031,0.004,0.069
3,Lat,Extratropical,47.75,49.75,49.37,0.000,0.00,0.000,0.000,0.000,0.000


In [15]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "Lon")

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,Lon,All,-54.80,-50.795,-53.850,0.0,0.000,0.00,0.151,0.000,0.015
1,Lon,Tropical,-57.91,-54.320,-56.455,0.0,0.000,0.00,0.695,0.000,0.001
2,Lon,Transitioning,-50.30,-47.960,-51.040,0.0,0.529,0.02,0.599,0.007,0.469
3,Lon,Extratropical,-41.85,-36.240,-37.390,0.0,0.000,0.00,0.000,0.000,0.000


In [19]:
ET_Diff_Signif(Control_ET, RCP45_ET, RCP85_ET, "SLP")

,Variable,Snapshot,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,SLP,Genesis,1006.340,1005.825,1006.915,0.187,0.692,0.072,0.840,0.253,0.432
1,SLP,Peak,961.430,959.980,956.765,0.140,0.051,0.111,0.033,0.658,0.216
2,SLP,Trop Peak,969.110,965.985,965.020,0.351,0.194,0.243,0.131,0.501,0.437
3,SLP,ET Begin,986.565,985.780,985.630,0.810,0.758,0.562,0.636,0.850,0.585
4,SLP,ET Complete,992.070,989.960,988.960,0.156,0.021,0.136,0.019,0.206,0.022


In [20]:
ET_Diff_Signif(Control_ET, RCP45_ET, RCP85_ET, "Lat")

,Variable,Snapshot,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,Lat,Genesis,22.16,25.105,25.800,0.030,0.034,0.267,0.046,0.148,0.022
1,Lat,Peak,34.13,37.635,37.240,0.009,0.030,0.038,0.026,0.020,0.049
2,Lat,Trop Peak,31.56,34.065,34.335,0.015,0.007,0.006,0.003,0.007,0.004
3,Lat,ET Begin,38.33,39.060,39.005,0.511,0.287,0.235,0.211,0.258,0.388
4,Lat,ET Complete,46.23,47.890,46.970,0.074,0.087,0.036,0.086,0.067,0.345


In [21]:
ET_Diff_Signif(Control_ET, RCP45_ET, RCP85_ET, "Lon")

,Variable,Snapshot,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,Lon,Genesis,-49.82,-48.550,-55.255,0.286,0.112,0.332,0.023,0.645,0.102
1,Lon,Peak,-57.85,-52.400,-56.045,0.029,0.105,0.053,0.422,0.012,0.370
2,Lon,Trop Peak,-61.03,-56.810,-58.950,0.011,0.207,0.051,0.536,0.016,0.112
3,Lon,ET Begin,-57.44,-51.485,-54.255,0.009,0.234,0.039,0.146,0.006,0.186
4,Lon,ET Complete,-44.12,-39.935,-40.630,0.001,0.074,0.028,0.145,0.048,0.147


In [22]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "B")

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,B,All,4.250,5.47,6.810,0.000,0.000,0.001,0.000,0.000,0.000
1,B,Tropical,-0.030,0.78,1.580,0.000,0.000,0.001,0.000,0.000,0.000
2,B,Transitioning,24.185,23.45,23.655,0.285,0.004,0.619,0.088,0.224,0.303
3,B,Extratropical,37.770,37.91,39.100,0.022,0.002,0.707,0.714,1.000,0.423


In [23]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "VLT")

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,VLT,All,134.00,125.27,120.53,0.000,0.000,0.000,0.000,0.002,0.000
1,VLT,Tropical,165.83,160.23,159.41,0.000,0.000,0.000,0.000,0.117,0.042
2,VLT,Transitioning,129.84,120.08,112.94,0.036,0.001,0.252,0.013,0.171,0.006
3,VLT,Extratropical,-78.32,-77.98,-75.52,0.723,0.214,0.641,0.338,0.840,0.366


In [24]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "VUT")

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),MW Test (RCP4.5 - Control),MW Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,VUT,All,22.080,23.805,18.17,0.059,0.000,0.329,0.006,0.350,0.068
1,VUT,Tropical,42.050,46.270,43.67,0.002,0.025,0.741,0.198,0.180,0.572
2,VUT,Transitioning,43.845,62.560,66.11,0.000,0.022,0.003,0.004,0.167,0.067
3,VUT,Extratropical,-126.780,-117.385,-121.79,0.045,0.287,0.018,0.982,0.035,0.351
